In [0]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import mlflow
import mlflow.sklearn

print("✅ Imports OK")

✅ Imports OK


In [0]:
# Leer desde Unity Catalog (no /mnt/)
nyctaxi_silver = spark.read.table("bigdata_final.taxis.silver")

df = nyctaxi_silver.select(
    "fare_per_mile", "time_period", "trip_type",
    "pickup_hour", "pickup_dayofweek", "vendor_id",
    "payment_type", "passenger_count"
).dropna().sample(fraction=0.82, seed=42).toPandas()

print(f"📦 Registros: {len(df):,}")

📦 Registros: 4,063,390


In [0]:
p33 = df["fare_per_mile"].quantile(0.33)
p66 = df["fare_per_mile"].quantile(0.66)

df["fare_label"] = df["fare_per_mile"].apply(
    lambda x: 0 if x <= p33 else (1 if x <= p66 else 2)
)

cat_cols = ["time_period", "trip_type", "vendor_id", "payment_type"]
num_cols = ["pickup_hour", "pickup_dayofweek", "passenger_count"]

for c in cat_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))

print(f"🏷️  Low: <${p33:.2f} | Mid: ${p33:.2f}–${p66:.2f} | High: >${p66:.2f}")
print(df["fare_label"].value_counts().sort_index())

🏷️  Low: <$3.48 | Mid: $3.48–$4.76 | High: >$4.76
fare_label
0    1357740
1    1326267
2    1379383
Name: count, dtype: int64


In [0]:
from sklearn.model_selection import train_test_split

X = df[cat_cols + num_cols]
y = df["fare_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"🏋️  Train: {len(X_train):,} | 🧪 Test: {len(X_test):,}")

mlflow.set_experiment("/Users/0234976@up.edu.mx/dynamic_pricing")

with mlflow.start_run(run_name="LR_sklearn"):
    lr = LogisticRegression(
        max_iter=500,
        C=5.0,
        solver="lbfgs"
    )
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="weighted")

    mlflow.log_param("model", "LogisticRegression_sklearn")
    mlflow.log_param("C", 5.0)
    mlflow.log_param("max_iter", 500)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1", f1)
    # Create signature for UC registration
    signature = mlflow.models.infer_signature(X_train, lr.predict(X_train))
    input_example = X_train.head(5)
    
    mlflow.sklearn.log_model(
        lr, 
        "model",
        signature=signature,
        input_example=input_example
    )

    print(f"✅ LR sklearn → Accuracy: {acc:.4f} | F1: {f1:.4f}")

🏋️  Train: 3,250,712 | 🧪 Test: 812,678


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/03 04:06:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-253d6731-2274.cloud.databricks.com/ml/experiments/3305854452753307/models/m-1d6d8c57132b4efcba286

✅ LR sklearn → Accuracy: 0.5315 | F1: 0.4890


In [0]:
import pickle

with open("/Volumes/bigdata_final/taxis/mlflow_tmp/lr_model.pkl", "wb") as f:
    pickle.dump(lr, f)

print("🏆 Modelo guardado")
print("🏁 ML completo")

🏆 Modelo guardado
🏁 ML completo


In [0]:
mlflow.set_registry_uri("databricks-uc")

model_uri = f"runs:/{mlflow.last_active_run().info.run_id}/model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name="bigdata_final.taxis.dynamic_pricing_lr"
)

print(f"✅ Modelo registrado → versión {registered.version}")

Successfully registered model 'bigdata_final.taxis.dynamic_pricing_lr'.
2026/06/03 04:10:44 WARNING mlflow.tracking._model_registry.fluent: Run with id 317607845505428883ccd1cd45d2e8c3 has no artifacts at artifact path 'model', registering model based on models:/m-1d6d8c57132b4efcba286692d7cf1491 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

✅ Modelo registrado → versión 1


🔗 Created version '1' of model 'bigdata_final.taxis.dynamic_pricing_lr': https://dbc-253d6731-2274.cloud.databricks.com/explore/data/models/bigdata_final/taxis/dynamic_pricing_lr/version/1?o=7474658958319842


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

client.set_registered_model_alias(
    name="bigdata_final.taxis.dynamic_pricing_lr",
    alias="Production",
    version=registered.version
)

print(f"✅ Versión {registered.version} → alias: Production")

✅ Versión 1 → alias: Production


In [0]:
import requests

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

endpoint_url = "https://dbc-253d6731-2274.cloud.databricks.com/serving-endpoints/dynamic_pricing_endpoint/invocations"

data = {
    "dataframe_records": [
        {
            "time_period": 0, "trip_type": 1, "pickup_hour": 8,
            "pickup_dayofweek": 2, "vendor_id": 0,
            "payment_type": 1, "passenger_count": 1
        },
        {
            "time_period": 2, "trip_type": 0, "pickup_hour": 2,
            "pickup_dayofweek": 6, "vendor_id": 1,
            "payment_type": 0, "passenger_count": 2
        },
        {
            "time_period": 1, "trip_type": 3, "pickup_hour": 18,
            "pickup_dayofweek": 1, "vendor_id": 0,
            "payment_type": 1, "passenger_count": 3
        }
    ]
}

response = requests.post(
    endpoint_url,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    },
    json=data
)

labels = {0: "Low 🟢", 1: "Mid 🟡", 2: "High 🔴"}
predictions = response.json()["predictions"]

print("══ Predicciones Dynamic Pricing ══")
for i, pred in enumerate(predictions):
    print(f"  Viaje {i+1}: {labels[pred]}")

══ Predicciones Dynamic Pricing ══
  Viaje 1: High 🔴
  Viaje 2: High 🔴
  Viaje 3: Mid 🟡


In [0]:
# ── RANGOS DE PRECIO REALES ───────────────────────────────────────────
# Leer Silver para calcular los percentiles reales
df_rangos = spark.read.table("bigdata_final.taxis.silver").toPandas()

p33 = float(df_rangos['total_amount'].quantile(0.33))
p67 = float(df_rangos['total_amount'].quantile(0.67))
p_min = float(df_rangos['total_amount'].min())
p_max = float(df_rangos['total_amount'].max())

print("=" * 50)
print("💰 RANGOS DE PRECIO")
print("=" * 50)
print(f"  🟢 Tarifa Baja:  ${p_min:.2f}  →  ${p33:.2f}")
print(f"  🟡 Tarifa Media: ${p33:.2f}  →  ${p67:.2f}")
print(f"  🔴 Tarifa Alta:  ${p67:.2f}  →  ${p_max:.2f}")
print("=" * 50)

In [0]:
import mlflow.pyfunc

# ── WRAPPER ───────────────────────────────────────────────────────────
class PricingWrapper(mlflow.pyfunc.PythonModel):

    def __init__(self, model, p_min, p33, p67, p_max):
        self.model = model
        self.labels = {
            0: f"Tarifa Baja  (${p_min:.2f} - ${p33:.2f})",
            1: f"Tarifa Media (${p33:.2f} - ${p67:.2f})",
            2: f"Tarifa Alta  (${p67:.2f} - ${p_max:.2f})"
        }

    def predict(self, context, model_input):
        preds = self.model.predict(model_input)
        return [self.labels[int(p)] for p in preds]


# Crear el wrapper con el modelo ya entrenado (lr = tu modelo actual)
wrapper = PricingWrapper(lr, p_min, p33, p67, p_max)

# ── REGISTRAR EN MLFLOW ───────────────────────────────────────────────
mlflow.set_registry_uri("databricks-uc")
MODEL_NAME = "bigdata_final.taxis.dynamic_pricing_lr"

with mlflow.start_run(run_name="dynamic_pricing_con_etiquetas"):

    mlflow.pyfunc.log_model(
        artifact_path         = "model",
        python_model          = wrapper,
        registered_model_name = MODEL_NAME
    )

    print("✅ Modelo con etiquetas registrado en MLflow")
    print(f"\nModelo: {MODEL_NAME}")
    print(f"\nEtiquetas configuradas:")
    for k, v in wrapper.labels.items():
        print(f"  {k} → {v}")

In [0]:
# ── PRUEBA LOCAL DEL WRAPPER ──────────────────────────────────────────
import pandas as pd

# Simular el mismo input que mandas desde Postman
test_input = pd.DataFrame([{
    "time_period": 0,
    "trip_type": 1,
    "pickup_hour": 8,
    "pickup_dayofweek": 2,
    "vendor_id": 0,
    "payment_type": 1,
    "passenger_count": 1
}])

resultado = wrapper.predict(None, test_input)
print("🧪 Prueba local del wrapper:")
print(f"   Input:  time_period=0, trip_type=1, pickup_hour=8")
print(f"   Output: {resultado[0]}")
print("\n✅ El wrapper funciona correctamente")